# ETL from OMOP
This notebook uses the SQL query used to create cohorts in the IDEA4RC project. The
First part of the notebook is to make groups of patient IDs which in IDEA4RC is done
by the cohort builder.

The cohorts (stored in Pandas dataframes) are then exported into parquet files to be 
used by the mock client

In [ ]:
from pkg_resources import importlib

# from ohdsi import common
from ohdsi.database_connector import connect
from ohdsi import database_connector
from ohdsi import common

## OMOP database
### Get connection details

### Connect to the database

In [ ]:
conn = connect(
    dbms = "postgresql",
    connection_string = "jdbc:postgresql://localhost:5432/omopdb",
    user = "ohdsi",
    password = "ohdsi",
)

## Extract the cohort data

In [ ]:
sessions = importlib.import_module("v6-sessions")

In [ ]:
# rps_cohort
sql = "SELECT person_id FROM omopcdm.measurement WHERE measurement_concept_id in (36770706)"
df = database_connector.query_sql(conn, sql)
df = common.convert_from_r(df)
print(df)
pts_RPS = list(df.person_id.values)
print(pts_RPS)

rps_cohort_original = sessions.cohort.__create_cohort_dataframe(
    connection=conn,
    patient_ids=pts_RPS,
    features="sarcoma"
)
rps_cohort = rps_cohort_original.to_pandas()

# pelvis_cohort
sql = "SELECT person_id FROM omopcdm.measurement WHERE measurement_concept_id in (36768752)"
df = database_connector.query_sql(conn, sql)
df = common.convert_from_r(df)
pts_Pelvis = list(df.person_id.values)

pelvis_cohort_original = sessions.cohort.__create_cohort_dataframe(
    connection=conn,
    patient_ids=pts_Pelvis,
    features="sarcoma"
)
pelvis_cohort = pelvis_cohort_original.to_pandas()

# rps_pelvis_cohort
sql = "SELECT person_id FROM omopcdm.measurement WHERE measurement_concept_id in (36770706, 36768752)"
df = database_connector.query_sql(conn, sql)
df = common.convert_from_r(df)
pts_RPS_Pelvis = list(df.person_id.values)

rps_pelvis_cohort_original = sessions.cohort.__create_cohort_dataframe(
    connection=conn,
    patient_ids=pts_RPS_Pelvis,
    features="sarcoma"
)
rps_pelvis_cohort = rps_pelvis_cohort_original.to_pandas()


print(f'RPS cohort: {len(rps_cohort)}')
print(f'Pelvis cohort: {len(pelvis_cohort)}')
print(f'RPS + Pelvis cohort: {len(rps_pelvis_cohort)}')

In [ ]:
rps_cohort.columns

In [ ]:
rps_cohort.dtypes

In [ ]:
# export all dataframes to parquet
rps_cohort.to_parquet("rps_cohort.parquet")
pelvis_cohort.to_parquet("pelvis_cohort.parquet")
rps_pelvis_cohort.to_parquet("rps_pelvis_cohort.parquet")